In [1]:
import pandas as pd
import pyodbc
import matplotlib.pyplot as plt

In [2]:
mdb_path = r"C:\Users\Jouke\Desktop\emergo_dataset\DemoDataBetsyBike.mdb"

conn_str = (
    r"Driver={Microsoft Access Driver (*.mdb, *.accdb)};"
    fr"DBQ={mdb_path};"
)
conn = pyodbc.connect(conn_str)

In [7]:
tables = [t.table_name for t in conn.cursor().tables(tableType='TABLE')]
tables

['Address',
 'Customer',
 'Employee',
 'Person',
 'Product',
 'ProductCategory',
 'ProductModel',
 'ProductSubcategory',
 'ProductVendor',
 'SalesOrderDetail',
 'SalesOrderHeader',
 'SalesTerritory',
 'SalesTerritoryHistory',
 'Store',
 'Vendor']

In [8]:
df = pd.read_sql("SELECT * FROM SalesOrderHeader", conn)
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

# Controleer of elk jaar volledig is (12 maanden)
year_month_counts = df.groupby([df['OrderDate'].dt.year, df['OrderDate'].dt.month]).size().unstack(fill_value=0)
incomplete_years = year_month_counts[year_month_counts.astype(bool).sum(axis=1) < 12]

print("Aantal maanden per jaar:")
print(year_month_counts)
print("\nOnvolledige jaren:")
print(incomplete_years)


C:\Users\Jouke\AppData\Local\Temp\ipykernel_46652\2262103706.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM SalesOrderHeader", conn)


Aantal maanden per jaar:
OrderDate    1     2     3     4     5     6    7     8     9     10    11  \
OrderDate                                                                    
2006          0     0     0     0     0     0  184   231   206   201   259   
2007        228   250   263   244   299   282  325   420   309   302   326   
2008        309   404   378   368   469   423  609  1760  1783  1779  1889   
2009       1946  2032  2109  2128  2386  2374    0     0     0     0     0   

OrderDate    12  
OrderDate        
2006        298  
2007        444  
2008       2272  
2009          0  

Onvolledige jaren:
OrderDate    1     2     3     4     5     6    7    8    9    10   11   12
OrderDate                                                                  
2006          0     0     0     0     0     0  184  231  206  201  259  298
2009       1946  2032  2109  2128  2386  2374    0    0    0    0    0    0


In [16]:
import pandas as pd

# Data inlezen
df = pd.read_sql("SELECT * FROM SalesOrderHeader", conn)
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

# Groepeer per jaar en maand
year_month_counts = (
    df.groupby([df['OrderDate'].dt.year, df['OrderDate'].dt.month])
      .size()
      .unstack(fill_value=0)
)

# Maandnummers → maandnamen
year_month_counts.columns = [
    pd.to_datetime(str(m), format='%m').strftime('%b') for m in year_month_counts.columns
]

# Zoek onvolledige jaren (< 12 maanden)
incomplete_years = year_month_counts[year_month_counts.astype(bool).sum(axis=1) < 12]

# Print zonder truncatie
print("Aantal maanden per jaar:")
print(year_month_counts.to_string())

print("\nOnvolledige jaren:")
print(incomplete_years.to_string())


C:\Users\Jouke\AppData\Local\Temp\ipykernel_46652\1565683273.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM SalesOrderHeader", conn)


Aantal maanden per jaar:
            Jan   Feb   Mar   Apr   May   Jun  Jul   Aug   Sep   Oct   Nov   Dec
OrderDate                                                                       
2006          0     0     0     0     0     0  184   231   206   201   259   298
2007        228   250   263   244   299   282  325   420   309   302   326   444
2008        309   404   378   368   469   423  609  1760  1783  1779  1889  2272
2009       1946  2032  2109  2128  2386  2374    0     0     0     0     0     0

Onvolledige jaren:
            Jan   Feb   Mar   Apr   May   Jun  Jul  Aug  Sep  Oct  Nov  Dec
OrderDate                                                                  
2006          0     0     0     0     0     0  184  231  206  201  259  298
2009       1946  2032  2109  2128  2386  2374    0    0    0    0    0    0


In [18]:
import pandas as pd

# Inlezen & datumkolom
df = pd.read_sql("SELECT * FROM SalesOrderHeader", conn)
df['OrderDate'] = pd.to_datetime(df['OrderDate'])

# FiscalYear: juli–juni
df['FiscalYear'] = df['OrderDate'].apply(lambda d: d.year if d.month >= 7 else d.year - 1)

# FiscalMonthIndex: Jul=1 ... Jun=12
df['FiscalMonthIndex'] = df['OrderDate'].dt.month.apply(lambda m: ((m - 7) % 12) + 1)

# Pivot in fiscale volgorde
fiscal_counts = (
    df.groupby(['FiscalYear', 'FiscalMonthIndex'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=range(1,13))  # 1..12 = Jul..Jun
)

# Maandnamen Jul..Jun
month_names = ['Jul','Aug','Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr','May','Jun']
fiscal_counts.columns = month_names

print("Aantal maanden per fiscaal jaar (juli–juni):")
print(fiscal_counts.to_string())

# Optioneel: check op onvolledige fiscale jaren (<12 maanden met data)
incomplete_fy = fiscal_counts[(fiscal_counts > 0).sum(axis=1) < 12]
print("\nOnvolledige fiscale jaren:")
print(incomplete_fy.to_string())


C:\Users\Jouke\AppData\Local\Temp\ipykernel_46652\3484100698.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM SalesOrderHeader", conn)


Aantal maanden per fiscaal jaar (juli–juni):
            Jul   Aug   Sep   Oct   Nov   Dec   Jan   Feb   Mar   Apr   May   Jun
FiscalYear                                                                       
2006        184   231   206   201   259   298   228   250   263   244   299   282
2007        325   420   309   302   326   444   309   404   378   368   469   423
2008        609  1760  1783  1779  1889  2272  1946  2032  2109  2128  2386  2374

Onvolledige fiscale jaren:
Empty DataFrame
Columns: [Jul, Aug, Sep, Oct, Nov, Dec, Jan, Feb, Mar, Apr, May, Jun]
Index: []
